In [1]:
# ─── ЯЧЕЙКА 1: Установка ─────────────────────────────────────────────────────
import sys
!{sys.executable} -m pip install rank-bm25


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ─── ЯЧЕЙКА 2: Создание БД и таблицы команд ──────────────────────────────────
import sqlite3
import json

DB_PATH = r"D:\bogdanov\PyProjects\Agents_project\commands.db"

def get_connection():
    return sqlite3.connect(DB_PATH)

def init_db():
    with get_connection() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS commands (
                name          TEXT PRIMARY KEY,
                command       TEXT NOT NULL,
                functionality TEXT NOT NULL,
                example       TEXT NOT NULL,
                usage_example TEXT NOT NULL
            )
        """)
        conn.commit()
    print("DB ready:", DB_PATH)

init_db()

DB ready: D:\bogdanov\PyProjects\Agents_project\commands.db


In [3]:
# ─── ЯЧЕЙКА 3: Заполнение таблицы команд ─────────────────────────────────────

#TODO: Добавить поле ожидания ответа (пример ожидания движка)
import json

COMMANDS = [
    {
        "name": "extract",
        "command": {"action": "extract", "agents": [], "prompts": {}},
        "functionality": "Get information from one or more agents by sending prompts and receiving answers",
        "example": "What does FinanceAgent know about Q3 revenue? / Узнай у агента про продажи",
        "usage_example": json.dumps({
            "input":  {"action": "extract", "agents": ["FinanceAgent"], "prompts": {"FinanceAgent": "What were Q3 revenues?"}},
            "output": {"action": "extract", "agents": ["FinanceAgent"], "answers": {"FinanceAgent": "Q3 revenue was $4.2M, up 12% YoY."}}
        }, ensure_ascii=False)
    },
    {
        "name": "add",
        "command": {"action": "add", "agent": "", "data": {}},
        "functionality": "Add new data or knowledge to an agent",
        "example": "Save this report to ResearchAgent / Добавь новый факт в агента",
        "usage_example": json.dumps({
            "input":  {"action": "add", "agent": "ResearchAgent", "data": {"topic": "AI trends 2025", "content": "LLMs are growing rapidly"}},
            "output": {"action": "add", "agent": "ResearchAgent", "status": "ok", "added_id": "rec_004"}
        }, ensure_ascii=False)
    },
    {
        "name": "delete",
        "command": {"action": "delete", "agent": "", "target": ""},
        "functionality": "Delete data or entry from an agent",
        "example": "Remove outdated market data from FinanceAgent / Удали устаревшие данные",
        "usage_example": json.dumps({
            "input":  {"action": "delete", "agent": "FinanceAgent", "target": "market_data_2022"},
            "output": {"action": "delete", "agent": "FinanceAgent", "status": "ok", "deleted": "market_data_2022"}
        }, ensure_ascii=False)
    },
    {
        "name": "edit",
        "command": {"action": "edit", "agent": "", "target": "", "data": {}},
        "functionality": "Modify or update existing data inside an agent",
        "example": "Update the project plan in PlanningAgent / Измени запись в агенте",
        "usage_example": json.dumps({
            "input":  {"action": "edit", "agent": "PlanningAgent", "target": "project_plan_v1", "data": {"deadline": "2025-09-01"}},
            "output": {"action": "edit", "agent": "PlanningAgent", "status": "ok", "updated": "project_plan_v1"}
        }, ensure_ascii=False)
    },
    {
        "name": "load",
        "command": {"action": "load", "agent": "", "file_path": ""},
        "functionality": "Load a file (pdf, csv, txt, json) into an agent for processing",
        "example": "Load report.pdf into FinanceAgent / Загрузи файл в агента",
        "usage_example": json.dumps({
            "input":  {"action": "load", "agent": "FinanceAgent", "file_path": "D:/reports/q3_report.pdf"},
            "output": {"action": "load", "agent": "FinanceAgent", "status": "ok", "chunks_loaded": 42}
        }, ensure_ascii=False)
    },
    {
        "name": "consolidate",
        "command": {"action": "consolidate", "source_agents": [], "target_agent": ""},
        "functionality": "Merge two or more agents into one new agent and remove originals",
        "example": "Merge Agent1 and Agent2 into Agent3 / Объедини двух агентов в одного",
        "usage_example": json.dumps({
            "input":  {"action": "consolidate", "source_agents": ["FinanceAgent", "BudgetAgent"], "target_agent": "FinancialAgent"},
            "output": {"action": "consolidate", "status": "ok", "created": "FinancialAgent", "removed": ["FinanceAgent", "BudgetAgent"]}
        }, ensure_ascii=False)
    },
    {
        "name": "split",
        "command": {"action": "split", "source_agent": "", "target_agents": []},
        "functionality": "Split one agent into two specialized agents and remove original",
        "example": "Split Agent3 into Agent1 and Agent2 / Раздели агента на двух специализированных",
        "usage_example": json.dumps({
            "input":  {"action": "split", "source_agent": "GeneralAgent", "target_agents": ["AnalyticsAgent", "ReportingAgent"]},
            "output": {"action": "split", "status": "ok", "created": ["AnalyticsAgent", "ReportingAgent"], "removed": "GeneralAgent"}
        }, ensure_ascii=False)
    },
]


def upsert_commands(commands: list):
    with get_connection() as conn:
        for c in commands:
            conn.execute("""
                INSERT INTO commands (name, command, functionality, example, usage_example)
                VALUES (?, ?, ?, ?, ?)
                ON CONFLICT(name) DO UPDATE SET
                    command       = excluded.command,
                    functionality = excluded.functionality,
                    example       = excluded.example,
                    usage_example = excluded.usage_example
            """, (
                c["name"],
                json.dumps(c["command"],       ensure_ascii=False),
                c["functionality"],
                c["example"],
                c["usage_example"]
            ))
        conn.commit()
    print(f"Upserted {len(commands)} commands")


def load_all_commands() -> list:
    with get_connection() as conn:
        rows = conn.execute(
            "SELECT name, command, functionality, example, usage_example FROM commands"
        ).fetchall()
    return [
        {
            "name":          r[0],
            "command":       json.loads(r[1]),
            "functionality": r[2],
            "example":       r[3],
            "usage_example": json.loads(r[4])
        }
        for r in rows
    ]


upsert_commands(COMMANDS)
print("Total commands:", len(load_all_commands()))

Upserted 7 commands
Total commands: 7


In [5]:
# ─── ЯЧЕЙКА 4: BM25 индекс поверх SQLite ─────────────────────────────────────
from rank_bm25 import BM25Okapi
import re

_bm25_index  = None
_bm25_corpus = []  # команды в том же порядке что индекс


def tokenize(text: str) -> list:
    return re.findall(r'\w+', text.lower())


def build_bm25_index():
    global _bm25_index, _bm25_corpus

    _bm25_corpus = load_all_commands()

    # Документ для каждой команды = functionality + example
    tokenized = [
        tokenize(f"{c['functionality']} {c['example']}")
        for c in _bm25_corpus
    ]

    _bm25_index = BM25Okapi(tokenized)
    print(f"BM25 index built: {len(_bm25_corpus)} commands")


def retrieve_commands(user_request: str, top_k: int = 2) -> list:
    """
    Ищет top_k команд по BM25.
    Автоматически перестраивает индекс если он не инициализирован.
    """
    global _bm25_index

    if _bm25_index is None:
        build_bm25_index()

    tokens = tokenize(user_request)
    scores = _bm25_index.get_scores(tokens)

    top_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:top_k]

    return [
        {**_bm25_corpus[i], "score": round(float(scores[i]), 4)}
        for i in top_indices
    ]


build_bm25_index()

BM25 index built: 7 commands


In [6]:
# ─── ЯЧЕЙКА 5: Тест retrieval ─────────────────────────────────────────────────

test_queries = [
    "узнай у агента про финансы",
    "загрузи файл отчёта в агента",
    "объедини двух агентов",
    "удали устаревшие данные",
    "раздели агента на два",
    "добавь новые данные",
]

for q in test_queries:
    found = retrieve_commands(q, top_k=1)
    print(f"Query:   {q}")
    print(f"Command: {found[0]['name']} (score: {found[0]['score']})")
    print()

Query:   узнай у агента про финансы
Command: extract (score: 4.2502)

Query:   загрузи файл отчёта в агента
Command: load (score: 3.4691)

Query:   объедини двух агентов
Command: consolidate (score: 3.5098)

Query:   удали устаревшие данные
Command: delete (score: 4.8612)

Query:   раздели агента на два
Command: split (score: 3.178)

Query:   добавь новые данные
Command: delete (score: 1.6204)

